# RL Trading Agent — Population-Based Training

**Colab Pro runtime required.** First cell detects GPU instance. Checkpoints save to Google Drive for crash recovery.

- Phase 1: 10 agents, 100 generations (~8 hours on T4)
- Phase 2: 20 agents, 500 generations (~40 hours)

In [1]:
import os
import platform
import subprocess

# Detect runtime environment
if os.path.exists("/content"):
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True)
    gpu_info = gpu.stdout.strip() if gpu.returncode == 0 else "No GPU detected"
    print(f"Runtime: Google Colab")
    print(f"GPU: {gpu_info}")
    print(f"RAM: {os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024**3):.1f} GB")
else:
    raise RuntimeError(
        "NOT running on Colab! This notebook must run on a Colab runtime.\n"
        "In VS Code: Select Kernel -> pick your Colab Pro runtime."
    )

Runtime: Google Colab
GPU: NVIDIA A100-SXM4-80GB, 81920 MiB
RAM: 167.1 GB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!git clone https://github.com/putratogatorop/AI-Finance.git /content/AI-Finance 2>/dev/null || (cd /content/AI-Finance && git pull && git checkout feat/rl-gym-environment)
%cd /content/AI-Finance/services/python
!pip install -q torch numpy pandas

In [ ]:
import numpy as np
from pathlib import Path
DRIVE_DATA = Path("/content/drive/MyDrive/ai-finance/rl_dataset.npz")
LOCAL_DATA = Path("/content/AI-Finance/data/features/rl_dataset.npz")
if DRIVE_DATA.exists():
    !cp "{DRIVE_DATA}" "{LOCAL_DATA}"
else:
    from google.colab import files
    print("Upload rl_dataset.npz:")
    uploaded = files.upload()
    !mkdir -p /content/AI-Finance/data/features/
    !mv rl_dataset.npz "{LOCAL_DATA}"
d = np.load(str(LOCAL_DATA), allow_pickle=True)
print(f"Dataset: {d['alt_features'].shape}")

In [ ]:
CKPT = "/content/drive/MyDrive/ai-finance/rl_checkpoints"
!python scripts/train_rl.py --data "{LOCAL_DATA}" --checkpoint {CKPT} --population 10 --generations 100 --episodes 10

In [ ]:
import json
from pathlib import Path
meta = Path(CKPT) / "metadata.json"
if meta.exists():
    with open(meta) as f: m = json.load(f)
    print(f"Gen: {m['generation']}, Best: {m['best_reward']:.4f}, Mean: {m['mean_reward']:.4f}")

In [ ]:
# !python scripts/train_rl.py --data "{LOCAL_DATA}" --checkpoint {CKPT} --population 20 --generations 500 --episodes 10

In [ ]:
# !python scripts/evaluate_rl.py --data "{LOCAL_DATA}" --checkpoint {CKPT}